# Lab 35: The Meaning Drain

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-35.ipynb)

**What you will do:** time how long it takes a word's meaning to drain away when you repeat it aloud for 60 seconds, compare your onset time with the 20-30 second range the book reports, and watch an open language model fall into its own repetition loop, then recover once a repetition penalty is switched on.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 35 you repeated a common word aloud, continuously, for 60 seconds. After about 20-30
seconds it began to sound strange, and by 60 seconds it could feel like a nonsense syllable
stripped of meaning — this is **semantic satiation**. The effect is temporary: after a short
pause, the word's meaning returns instantly.

## Record your results

For several different words, time how long you repeated each one before its meaning first
felt like it had drained away. Do this for a few words, since the exact moment is subjective
and will vary from word to word.

In [ ]:
# example data: replace with your own timings, in seconds
onsets_s = {"door": 24, "chair": 31, "really": 19, "the": 15}  # example data

onsets = pd.Series(onsets_s, name="seconds until meaning felt drained")
display(onsets.to_frame())
onsets.plot.barh(color="steelblue", figsize=(6, 2.4))
plt.xlabel("seconds"); plt.gca().invert_yaxis(); plt.show()

## Compare

Lab 35 reports that the effect typically begins after about 20-30 seconds of repetition.
Compare your own mean onset time with that range.

In [ ]:
mean_onset = onsets.mean()
print(f"Your mean onset: {mean_onset:.0f} s")
if 20 <= mean_onset <= 30:
    print("Inside the typical 20-30 second range Lab 35 reports.")
elif mean_onset < 20:
    print("Faster than typical -- short, very frequent function words such as 'the' can"
          " satiate especially quickly.")
else:
    print("Slower than typical -- some words resist satiation longer, and attention or"
          " distraction while repeating can also delay the effect.")

## The AI side

This is a hands-on version of the "AI extension" from the book: DistilGPT-2 is asked to keep
generating text from a repeated prompt, first with ordinary greedy decoding (always picking
the single most likely next token) and then with a **repetition penalty**, the engineered
fix language models use to avoid exactly this failure (Holtzman et al., 2020). Watch the
first output loop; the second should break out of the loop and move on to new words.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")
model.eval()

prompt = "The door. The door. The door."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

with torch.no_grad():
    looping = model.generate(input_ids, max_new_tokens=30, do_sample=False)
    fixed = model.generate(input_ids, max_new_tokens=30, do_sample=False, repetition_penalty=1.3)

results = pd.DataFrame({
    "condition": ["greedy decoding (no penalty)", "with repetition_penalty=1.3"],
    "generated continuation": [
        tokenizer.decode(looping[0], skip_special_tokens=True),
        tokenizer.decode(fixed[0], skip_special_tokens=True),
    ],
})
for _, row in results.iterrows():
    print(f"--- {row['condition']} ---\n{row['generated continuation']}\n")

If the unpenalised output keeps repeating "The door" (or falls into some other short loop),
it is producing the AI equivalent of a satiated word: technically well-formed, but carrying
no new information. The repetition penalty is not a biological process — it is an explicit
adjustment to next-token probabilities, switched on or off at will, whereas your own
satiation was an involuntary synaptic fatigue that you could not simply switch off, and that
reversed itself automatically after a short pause without any equivalent of a "penalty."

## Questions to think about

1. The book explains satiation as synaptic fatigue specifically in the pathway linking a word's phonological form to its meaning, not in the visual or motor systems used to read or say it. Does your experience of the visual version (staring at the written word rather than saying it aloud) support or complicate the idea that a single amodal "meaning" pathway is fatiguing?
2. In the AI Lab, does the repetition penalty change what DistilGPT-2 "knows" about the word "door," or only how it decides which word to output next? How is this similar to, or different from, the book's claim that your own satiation affects semantic access rather than lexical recognition (you can still read the word; you just cannot immediately access what it means)?
3. The book links semantic satiation to why political slogans lose force with repetition, but also notes that a brand-new coined term (like "doomscrolling") carries maximum impact on first exposure. Where would a slogan like this sit on the same dimension that the AI Lab's repetition_penalty controls?
4. After your pause in the original experiment, the word's meaning returned instantly. Is there anything in the AI Lab's set-up that behaves like this instant "reset," or does the model simply never accumulate any fatigue to reset in the first place?

## Challenge

Repeat a word you know well in a second language you speak, for 60 seconds, alongside its
English translation on a separate occasion. Does satiation set in at a similar speed for both,
or does one drain faster? This bears on whether the effect is tied to a word's sound pattern or
to the underlying concept it names.

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-35.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")